In [9]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Statsmodels for interpretable logistic regression output
import statsmodels.api as sm

# Scikit-learn
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, log_loss
)

In [3]:
df = pd.read_csv("../data/kaggle_b2_fraud_train_v3.csv")
df

,customer_id,account_id,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,...,internal_signal_5,internal_signal_6,internal_signal_7,internal_signal_8,terms_accepted_flag,partner_risk_indicator,manual_review_result,post_event_status_code,chargeback_resolution_time_days,legacy_partner_score
0,CUST_6O9Q8D4I36,ACC_TXXXTNEUVKFY,34,108,38635.01,544.0,20,60.92,80.16,4.9,...,0.39006,0.10963,0.55097,-0.56104,1,NaN,approve,0,7.9,NaN
1,CUST_FGUGTW230C,ACC_70VD7A4FFWCW,48,2,19912.97,703.0,21,112.11,571.12,0.3,...,0.03265,-0.40256,0.36218,0.86583,1,NaN,approve,0,5.5,NaN
2,CUST_8ZI3LCBZ0W,ACC_AF53381QSC0L,27,0,20326.87,720.0,25,73.61,492.57,4.6,...,-0.15637,0.57818,0.28902,-2.19864,1,NaN,approve,0,7.2,NaN
3,CUST_5MP3AR41CJ,ACC_U7WZGJ486LIV,45,49,38452.47,703.0,17,47.53,204.18,25.3,...,-1.02145,0.63908,-0.89190,-0.81592,1,NaN,approve,0,4.4,NaN
4,CUST_GNPL83JB0J,ACC_XW7DS3ED5J4Y,37,46,NaN,594.0,13,99.95,734.09,12.8,...,-0.65771,0.08020,0.17606,0.86739,1,NaN,approve,0,4.9,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159995,CUST_I81IW5SVRQ,ACC_UPDTFTYTSM0A,56,0,34775.62,727.0,21,51.72,226.11,3.8,...,-2.54086,-0.60747,0.23252,-0.06215,1,-1.535486,approve,0,1.0,NaN
159996,CUST_QT6DDEMKTJ,ACC_97NE0LBL5W9U,41,4,88617.57,770.0,18,NaN,171.07,15.1,...,0.34098,-1.78817,0.31788,0.51072,1,NaN,approve,0,7.4,NaN
159997,CUST_I0JS1GTS98,ACC_9JJ84W64Z7GX,30,2,41148.54,738.0,20,29.34,119.81,0.7,...,-1.28947,-0.32324,-0.06238,-0.99076,1,NaN,approve,3,6.6,NaN
159998,CUST_L7GUCJ3TFY,ACC_NGFXDR7HW1ZS,56,6,NaN,719.0,25,88.56,553.16,22.6,...,0.47179,-0.22090,-1.34239,-0.30513,1,NaN,approve,0,12.5,NaN


In [4]:
from statsmodels.stats import descriptivestats

df.describe()

,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,support_tickets_90d,chargebacks_12m,...,internal_signal_4,internal_signal_5,internal_signal_6,internal_signal_7,internal_signal_8,terms_accepted_flag,partner_risk_indicator,post_event_status_code,chargeback_resolution_time_days,legacy_partner_score
count,160000.000000,160000.000000,1.488140e+05,152048.000000,160000.000000,150453.000000,152077.000000,160000.000000,160000.000000,160000.000000,...,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.0,4751.000000,160000.000000,160000.000000,6264.000000
mean,38.239075,17.898356,3.761071e+04,670.067327,22.901775,61.626094,298.683078,12.019874,0.797850,0.049931,...,-0.000427,-0.000285,0.000559,-0.001698,-0.000445,1.0,-0.005003,0.257069,6.798126,-0.023656
std,11.751027,17.781320,2.693766e+04,59.985184,34.776594,41.795545,256.039318,12.033636,0.894715,0.223217,...,0.999285,0.997891,0.999006,1.000762,0.997346,0.0,1.006872,0.871035,5.542037,1.008418
min,-14.000000,-23.000000,-1.994171e+04,380.000000,5.000000,-49.675409,0.610000,0.000000,0.000000,0.000000,...,-5.166910,-4.159100,-4.845700,-5.101360,-4.465930,1.0,-4.067918,0.000000,0.000000,-3.819104
25%,30.000000,5.000000,2.204844e+04,630.000000,19.000000,30.970000,117.380000,3.500000,0.000000,0.000000,...,-0.675655,-0.670640,-0.672752,-0.673293,-0.672190,1.0,-0.685078,0.000000,3.400000,-0.713137
50%,38.000000,12.000000,3.194314e+04,670.000000,22.000000,52.540000,224.730000,8.300000,1.000000,0.000000,...,-0.002575,0.000560,0.002600,-0.004255,-0.002920,1.0,0.005115,0.000000,6.200000,-0.023162
75%,46.000000,25.000000,4.645079e+04,710.000000,25.000000,82.370000,402.170000,16.700000,1.000000,0.000000,...,0.673510,0.674070,0.675820,0.675828,0.671020,1.0,0.669542,0.000000,9.000000,0.658035
max,159.000000,120.000000,1.355012e+06,850.000000,1923.000000,417.100000,2994.200000,184.600000,7.000000,3.000000,...,4.183620,4.601760,4.631420,4.192480,4.878210,1.0,3.649050,4.000000,63.100000,3.607133


In [7]:
import pandas as pd

df_numeric = df.select_dtypes(exclude=['object', 'category'])

df_numeric = df_numeric.dropna(axis=1)

print("Colonnes restantes :", df_numeric.columns.tolist())



Colonnes restantes : ['age', 'tenure_months', 'num_transactions_30d', 'days_since_last_login', 'support_tickets_90d', 'chargebacks_12m', 'failed_payments_6m', 'is_vpn', 'num_devices_30d', 'is_new_device', 'postal_code', 'target_is_fraud', 'income_log', 'income_estimate_alt_eur', 'credit_score_norm', 'tx_amount_total_30d_eur', 'max_to_avg_ratio', 'internal_signal_1', 'internal_signal_2', 'internal_signal_3', 'internal_signal_4', 'internal_signal_5', 'internal_signal_6', 'internal_signal_7', 'internal_signal_8', 'terms_accepted_flag', 'post_event_status_code', 'chargeback_resolution_time_days']


In [12]:
X = df_numeric.drop(columns="target_is_fraud")
y = df_numeric["target_is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)

knn.fit(X_train_scaled, y_train)
y_pred = knn.predict(X_test_scaled)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy : 0.994875
Confusion Matrix:
 [[38731    19]
 [  186  1064]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     38750
           1       0.98      0.85      0.91      1250

    accuracy                           0.99     40000
   macro avg       0.99      0.93      0.95     40000
weighted avg       0.99      0.99      0.99     40000



In [13]:
df_test = pd.read_csv("../data/kaggle_b2_fraud_test_v3.csv")
df_test

,customer_id,account_id,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,...,internal_signal_5,internal_signal_6,internal_signal_7,internal_signal_8,terms_accepted_flag,partner_risk_indicator,manual_review_result,post_event_status_code,chargeback_resolution_time_days,legacy_partner_score
0,CUST_E5RX1BC9II,ACC_YFJ1HGWZBXY7,43,53,25587.55,639.0,24,106.61,781.55,7.3,...,-1.32827,-0.67095,0.96425,0.13556,1,NaN,approve,0,9.0,NaN
1,CUST_BHWIUKERGN,ACC_3GPLTB4O5TQR,22,5,45378.40,699.0,21,54.17,252.95,4.2,...,0.38501,0.98914,1.56030,1.05211,1,NaN,review,0,8.8,NaN
2,CUST_EXT9NA4CHU,ACC_BJV525XTFPU4,42,2,36643.70,765.0,17,44.40,238.79,7.0,...,-1.02340,-1.15396,-0.33549,-0.15284,1,NaN,approve,0,5.3,NaN
3,CUST_9FSJE5R1NY,ACC_3R73734P68KY,39,20,30283.30,573.0,29,38.30,321.44,28.3,...,0.86966,0.47447,0.41196,0.23284,1,NaN,review,2,0.5,NaN
4,CUST_GDQXMODBED,ACC_OACZ35SKXJOU,54,10,35294.22,624.0,21,70.93,540.48,21.7,...,-0.05352,-0.02331,-0.22379,-0.88609,1,NaN,approve,4,13.8,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,CUST_1OM9UCID91,ACC_LKZHMRZ96RKC,44,3,41531.58,622.0,24,42.99,214.60,7.3,...,0.00140,2.30414,-0.55719,1.37723,1,-0.4551,approve,0,8.1,NaN
39996,CUST_VDEY72BIZP,ACC_LS8VIDTNSLVW,48,12,36783.80,697.0,25,21.17,69.19,3.8,...,-0.00073,1.45201,-0.08652,-0.98155,1,NaN,approve,0,5.4,NaN
39997,CUST_UQEZ9KKIFG,ACC_4QLK8U7F3ZCS,49,28,55963.60,658.0,18,180.58,410.17,3.6,...,0.15541,-0.74079,1.37543,1.14303,1,NaN,approve,0,16.7,NaN
39998,CUST_IXX0BE9VQD,ACC_7Y9LSXFR8WS8,31,7,9638.61,683.0,19,129.93,643.13,13.9,...,3.02645,-0.63311,-0.00841,1.52853,1,NaN,review,4,1.6,NaN


In [15]:
y_test_pred = knn.predict(X_test_scaled) 


In [17]:
df_results = pd.DataFrame({
    "customer_id": df_test["customer_id"],  
    "target_is_fraud": y_test_pred          
})

print(df_results.head())

df_results.to_csv("../Preds/KNN_preds_Noah.csv", index=False)

       customer_id  target_is_fraud
0  CUST_E5RX1BC9II                0
1  CUST_BHWIUKERGN                1
2  CUST_EXT9NA4CHU                0
3  CUST_9FSJE5R1NY                0
4  CUST_GDQXMODBED                0
